# exp075_compact_tracker_pfbeam_feature_repro_guard PF/Beam train feature generation

Generate the reusable compact PF/Beam/likelihood-PF tracker feature frame from raw train data. This notebook writes the train feature CSV once; LightGBM training reads that output instead of regenerating an equivalent file.

## Contents

1. Setup and configuration
2. Raw train input check
3. PF/Beam train feature generation
4. Feature generation artifacts

## 1. Setup and configuration

In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config, get_nested
from compact_tracker_pfbeam_repro_guard import (
    TRACKER_TRAIN_FEATURES,
    OUTPUT_PREFIX,
    run_pfbeam_feature_generation,
)

def cfg_get(config, dotted_key, default=None):
    value = get_nested(config, dotted_key)
    return default if value is None else value


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Feature generation mode:", cfg_get(config, "feature_generation.mode"))
print("Raw data dir:", paths.raw_data_dir)
print("Output tracker file:", paths.artifacts_dir / TRACKER_TRAIN_FEATURES)
print("PF seeds:", cfg_get(config, "feature_generation.pf_seeds"))
print("PF particles:", cfg_get(config, "feature_generation.pf_particles"))


## 2. Raw train input check

In [ ]:
train_files = sorted(paths.train_data_dir.glob("*__horizontal_well.csv"))
typewell_files = sorted(paths.train_data_dir.glob("*__typewell.csv"))
print("Train horizontal wells:", len(train_files))
print("Train typewells:", len(typewell_files))
print("First train files:", [path.name for path in train_files[:5]])
if train_files:
    display(pd.read_csv(train_files[0], nrows=5))


## 3. PF/Beam train feature generation

In [ ]:
summary = run_pfbeam_feature_generation(
    data_dir=paths.raw_data_dir,
    output_dir=paths.artifacts_dir,
    n_jobs=int(cfg_get(config, "feature_generation.n_jobs", 8)),
    pf_seeds=int(cfg_get(config, "feature_generation.pf_seeds", 128)),
    pf_particles=int(cfg_get(config, "feature_generation.pf_particles", 500)),
    fast=bool(cfg_get(config, "feature_generation.fast", False)),
    use_gpu=str(cfg_get(config, "feature_generation.use_gpu", "auto")),
    max_wells=cfg_get(config, "feature_generation.max_wells"),
)
print(json.dumps(summary, indent=2))


## 4. Feature generation artifacts

In [ ]:
tracker_path = paths.artifacts_dir / TRACKER_TRAIN_FEATURES
summary_path = paths.artifacts_dir / f"{OUTPUT_PREFIX}_feature_generation_summary.json"
print("Tracker train features:", tracker_path, "exists=", tracker_path.exists())
print("Feature generation summary:", summary_path, "exists=", summary_path.exists())
preview = pd.read_csv(tracker_path, nrows=5, dtype={"id": str, "well": str})
print("Columns:", len(preview.columns))
display(preview)
